In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# # sample  =  pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
# # sample.to_csv('submission.csv', index=False)
# !pip uninstall -y transformers sentence-transformers

# Smart MCQ Solver Challenge — End-to-End Solution
### MAP@3 ranking of top-3 answers (A–E) for knowledge-based MCQs

**Pipeline overview**

| # | Model | Category | Idea |
|---|-------|----------|------|
| 1 | TF-IDF + PyTorch MLP | **Built from scratch** | No pretrained weights anywhere; learns purely from the ~2,000 training rows |
| 2 | DeBERTa-v3 (`AutoModelForMultipleChoice`) | **Pretrained, fine-tuned** | Transfer learning — pretrained language + world knowledge, adapted to this task |
| 3 | XGBoost on engineered similarity features (TF-IDF sim, Sentence-Transformer sim, lexical overlap, etc.) | **Additional model of choice** | Tree-based, feature-driven — a structurally different failure mode from 1 & 2 |
| — | Weighted ensemble of the three | **Final submission** | Combines probability outputs, tuned on a held-out validation split |

**Notebook structure**
1. Setup & data loading
2. EDA (light)
3. MAP@3 metric implementation
4. Train/validation split
5. Model 1 — from-scratch TF-IDF + MLP
6. Model 2 — pretrained DeBERTa-v3 fine-tuned as multiple-choice classifier
7. Model 3 — XGBoost on engineered similarity features
8. Ensembling + local MAP@3 evaluation
9. Final inference on `test.csv` + submission file
10. (Optional, commented out) Zero-shot LLM prompting extension

> Upload `train.csv`, `test.csv`, and `sample_submission.csv` to the Colab working directory (or mount Google Drive) before running.


##1. Setup

In [3]:
# Run this once per Colab session.
# ============================================================
# KAGGLE: INSTALL ONLY WHAT YOU NEED (NO UPGRADES)
# ============================================================
# Install specific versions WITHOUT upgrading system packages
!pip install -q transformers==4.41.2 datasets==2.20.0 accelerate==0.31.0 peft==0.11.1
!pip install -q sentence-transformers==2.7.0
!pip install -q scikit-learn==1.5.0 xgboost==2.0.3
!pip install -q wandb

# DO NOT upgrade numpy, torch, tensorflow, cuda, or RAPIDS packages

print("✅ Installation complete - using Kaggle's base packages")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires 

In [4]:
import os, re, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# this is now the errro , it is jsut a warning 


Using device: cuda


In [5]:
# ---- Paths: adjust if your files live elsewhere (e.g. Google Drive) ----
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge/"   # change to "/content/drive/MyDrive/mcq_challenge" if using Drive

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

OPTIONS = ["A", "B", "C", "D", "E"]
print(train.shape, test.shape, sample_submission.shape)
train.head()


(2000, 8) (500, 7) (500, 2)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## 2. Light EDA

In [6]:
print("Missing values (train):\n", train.isnull().sum())
print("\nAnswer class balance:\n", train["answer"].value_counts())
print("\nPrompt length stats (chars):\n", train["prompt"].str.len().describe())

Missing values (train):
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer class balance:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Prompt length stats (chars):
 count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt, dtype: float64


## 3. MAP@3 metric

For each question, if the true label appears at rank *k* (1-indexed) among our top-3 predictions,
the score for that question is `1/k`; if it doesn't appear in the top 3, the score is `0`.
The final metric is the mean over all questions.


In [7]:
def map_at_3(y_true, top3_preds):
    """
    y_true      : list/array of true labels, e.g. ['A','C',...]
    top3_preds  : list of lists, each an ordered top-3 prediction e.g. [['A','B','C'], ...]
    """
    scores = []
    for true_label, preds in zip(y_true, top3_preds):
        score = 0.0
        for rank, p in enumerate(preds[:3], start=1):
            if p == true_label:
                score = 1.0 / rank
                break
        scores.append(score)
    return float(np.mean(scores))

def probs_to_top3(prob_matrix, classes=OPTIONS):
    """prob_matrix: (n_samples, 5) array of class probabilities in the order of `classes`."""
    order = np.argsort(-prob_matrix, axis=1)  # descending
    top3 = [[classes[idx] for idx in row[:3]] for row in order]
    return top3


## 4. Train / validation split

We hold out 15% of the training data (stratified by answer label) purely for local MAP@3 evaluation
and ensemble-weight tuning. The final models are re-fit on the *full* training set before predicting on `test.csv`.


In [8]:
train_idx, val_idx = train_test_split(
    train.index, test_size=0.15, random_state=SEED, stratify=train["answer"]
)
tr_df  = train.loc[train_idx].reset_index(drop=True)
val_df = train.loc[val_idx].reset_index(drop=True)
print("train:", tr_df.shape, " val:", val_df.shape)

train: (1700, 8)  val: (300, 8)


## 5. Model 1 — Built From Scratch: TF-IDF + PyTorch MLP

No pretrained weights are used anywhere in this model. We:
1. Fit a TF-IDF vectorizer on `prompt + all options` from the training data only.
2. For each (prompt, option) pair, build a feature vector = `[tfidf(prompt) , tfidf(option) , elementwise-product]`
   (a lightweight, from-scratch analogue of an interaction feature — captures lexical overlap without any pretrained embeddings).
3. Train a small MLP classifier from scratch with cross-entropy loss over the 5 options.


In [9]:
def build_corpus(df):
    return pd.concat([df["prompt"], df["A"], df["B"], df["C"], df["D"], df["E"]]).astype(str).tolist()

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 2), stop_words="english")
tfidf.fit(build_corpus(tr_df))

def make_pair_features(df, vectorizer):
    """
    Returns X of shape (n_samples, 5, 3*D) -- one row of features per option, per question.
    D = tfidf dimensionality after an SVD-free direct approach would be huge, so instead we
    just use similarity + raw scores which is far more memory-friendly for a from-scratch MLP.
    """
    prompt_vecs = vectorizer.transform(df["prompt"].astype(str))
    feats = np.zeros((len(df), 5, 4), dtype=np.float32)  # 4 hand-built, from-scratch features per option
    for i, opt in enumerate(OPTIONS):
        opt_vecs = vectorizer.transform(df[opt].astype(str))
        sim = cosine_similarity(prompt_vecs, opt_vecs).diagonal()          # semantic-lexical overlap
        opt_len = df[opt].astype(str).str.len().values
        prompt_len = df["prompt"].astype(str).str.len().values
        len_ratio = opt_len / (prompt_len + 1)
        word_overlap = df.apply(
            lambda r, o=opt: len(set(str(r["prompt"]).lower().split()) & set(str(r[o]).lower().split())),
            axis=1
        ).values
        feats[:, i, 0] = sim
        feats[:, i, 1] = len_ratio
        feats[:, i, 2] = word_overlap
        feats[:, i, 3] = opt_len
    return feats

X_tr_raw  = make_pair_features(tr_df, tfidf)
X_val_raw = make_pair_features(val_df, tfidf)

# Normalize features (per-column) using train statistics only
mu, sigma = X_tr_raw.reshape(-1, 4).mean(0), X_tr_raw.reshape(-1, 4).std(0) + 1e-6
X_tr  = (X_tr_raw  - mu) / sigma
X_val = (X_val_raw - mu) / sigma

le = LabelEncoder().fit(OPTIONS)
y_tr  = le.transform(tr_df["answer"])
y_val = le.transform(val_df["answer"])

print(X_tr.shape, X_val.shape)

(1700, 5, 4) (300, 5, 4)


In [10]:
class MCQFeatureDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

class ScratchMLP(nn.Module):
    """From-scratch model: a small MLP that scores each of the 5 options."""
    def __init__(self, n_features=4, hidden=64):
        super().__init__()
        self.option_scorer = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        # x: (batch, 5, n_features) -> score per option -> (batch, 5)
        b, n_opts, n_feat = x.shape
        scores = self.option_scorer(x.reshape(-1, n_feat)).reshape(b, n_opts)
        return scores

train_ds = MCQFeatureDataset(X_tr, y_tr)
val_ds   = MCQFeatureDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)

model1 = ScratchMLP().to(DEVICE)
optimizer = torch.optim.Adam(model1.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS = 40
best_val_map3 = -1
best_state = None

for epoch in range(EPOCHS):
    model1.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model1(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    model1.eval()
    with torch.no_grad():
        val_logits = model1(torch.tensor(X_val, dtype=torch.float32).to(DEVICE))
        val_probs = F.softmax(val_logits, dim=1).cpu().numpy()
    top3 = probs_to_top3(val_probs)
    val_map3 = map_at_3(val_df["answer"].tolist(), top3)

    if val_map3 > best_val_map3:
        best_val_map3 = val_map3
        best_state = {k: v.clone() for k, v in model1.state_dict().items()}

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d} | train_loss={total_loss/len(train_ds):.4f} | val_MAP@3={val_map3:.4f}")

model1.load_state_dict(best_state)
print("\nModel 1 (from scratch) best val MAP@3:", round(best_val_map3, 4))

Epoch 01 | train_loss=1.5710 | val_MAP@3=0.5711
Epoch 05 | train_loss=1.4000 | val_MAP@3=0.6289
Epoch 10 | train_loss=1.3676 | val_MAP@3=0.6517
Epoch 15 | train_loss=1.3317 | val_MAP@3=0.6672
Epoch 20 | train_loss=1.3338 | val_MAP@3=0.6639
Epoch 25 | train_loss=1.3111 | val_MAP@3=0.6794
Epoch 30 | train_loss=1.3140 | val_MAP@3=0.6656
Epoch 35 | train_loss=1.3095 | val_MAP@3=0.6706
Epoch 40 | train_loss=1.2882 | val_MAP@3=0.6728

Model 1 (from scratch) best val MAP@3: 0.6806
